In [ ]:
Module_code = r'''
"""
Project  : 
"""
from __future__ import annotations
import os, math, random
from dataclasses import dataclass
from typing import Tuple

import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim_metric
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from mamba_ssm import Mamba

try:
    from pytorch_msssim import ssim
except ImportError:
    ssim = None
    

def seed_everything(seed: int = 123, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=False)
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# -----------------------------
# Utilities: Metrics & helpers
# -----------------------------

class EMA:
    """Exponential Moving Average for model parameters."""
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.detach().clone()
    @torch.no_grad()
    def update(self, model: nn.Module):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            assert name in self.shadow
            new_avg = self.decay * self.shadow[name] + (1.0 - self.decay) * param.detach()
            self.shadow[name] = new_avg.clone()

    @torch.no_grad()
    def store(self, model: nn.Module):
        """Save current parameters before swapping to EMA."""
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.detach().clone()

    @torch.no_grad()
    def copy_to(self, model: nn.Module):
        """Copy EMA weights into the model (for eval/save)."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                param.data.copy_(self.shadow[name].data)

    @torch.no_grad()
    def restore(self, model: nn.Module):
        """Restore original (non-EMA) weights after eval/save."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data.copy_(self.backup[name].data)


def calc_psnr(pred: np.ndarray, gt: np.ndarray, max_val: float = 1.0) -> float:
    """
    pred, gt: 2D numpy arrays in [0,1]
    returns PSNR in dB
    """
    mse = np.mean((pred - gt) ** 2)
    if mse == 0:
        return 100.0
    return 20.0 * math.log10(max_val) - 10.0 * math.log10(mse)


def to_y_channel(img: torch.Tensor) -> torch.Tensor:
    """
    Safely get a single-channel luminance-like tensor in [0,1].
    - If img is (N,1,H,W): return as is.
    - If img is (N,3,H,W): convert RGB -> Y (BT.601 approx).
    - Otherwise: if channel count is weird, just take first channel.
    """
    if img.dim() != 4:
        raise ValueError(f"Expected 4D tensor (N,C,H,W), got {img.shape}")

    c = img.size(1)

    if c == 1:
        # already single channel (e.g., CT slice)
        return img
    elif c >= 3:
        # use first 3 channels as RGB
        r = img[:, 0:1]
        g = img[:, 1:2]
        b = img[:, 2:3]
        # Y channel approximation (BT.601)
        y = 0.257 * r + 0.504 * g + 0.098 * b + 16 / 255.0
        return y
    else:
        # fallback, weird case (C=2 etc): take first channel and pretend it's Y
        return img[:, 0:1]


@torch.no_grad()
def batch_psnr_ssim_y(pred: torch.Tensor, gt: torch.Tensor) -> Tuple[float, float]:
    """
    Compute PSNR/SSIM per-image using a safe Y/single-channel version.
    Handles grayscale or RGB inputs.
    Returns (avg_psnr, avg_ssim) over batch.
    """
    # force to [0,1]
    pred_y = to_y_channel(pred).clamp(0, 1).detach().cpu().numpy()  # (N,1,H,W)
    gt_y   = to_y_channel(gt  ).clamp(0, 1).detach().cpu().numpy()  # (N,1,H,W)

    psnrs, ssims = [], []
    for i in range(pred_y.shape[0]):
        # skip rare corrupted batch entries
        if pred_y.shape[1] == 0 or gt_y.shape[1] == 0:
            continue

        p = pred_y[i, 0]  # (H,W)
        g = gt_y[i, 0]    # (H,W)

        psnrs.append(calc_psnr(p, g, 1.0))
        ssims.append(ssim_metric(g, p, data_range=1.0))

    if len(psnrs) == 0:
        return 0.0, 0.0

    return float(np.mean(psnrs)), float(np.mean(ssims))


# -----------------------------
# Network building blocks
# -----------------------------
#--------------------------------------------------------------
class HaarDWT2D(nn.Module):
    """
    Fixed 2D Haar DWT using conv2d stride=2.
    Input : x (N,C,H,W)
    Output: LL, LH, HL, HH each (N,C,H/2,W/2)
    """
    def __init__(self):
        super().__init__()

        # 2x2 Haar filters (normalized)
        ll = torch.tensor([[1.,  1.],
                           [1.,  1.]], dtype=torch.float32) / 2.0
        lh = torch.tensor([[1.,  1.],
                           [-1., -1.]], dtype=torch.float32) / 2.0
        hl = torch.tensor([[1., -1.],
                           [1., -1.]], dtype=torch.float32) / 2.0
        hh = torch.tensor([[1., -1.],
                           [-1., 1.]], dtype=torch.float32) / 2.0

        self.register_buffer("k_ll", ll.view(1, 1, 2, 2))
        self.register_buffer("k_lh", lh.view(1, 1, 2, 2))
        self.register_buffer("k_hl", hl.view(1, 1, 2, 2))
        self.register_buffer("k_hh", hh.view(1, 1, 2, 2))

    def forward(self, x: torch.Tensor):
        if x.dim() != 4:
            raise ValueError(f"HaarDWT2D expects (N,C,H,W), got {tuple(x.shape)}")
        N, C, H, W = x.shape
        if (H % 2 != 0) or (W % 2 != 0):
            raise ValueError(f"H and W must be even for Haar DWT. Got H={H}, W={W}")

        # repeat kernels per-channel and use groups=C
        k_ll = self.k_ll.repeat(C, 1, 1, 1)
        k_lh = self.k_lh.repeat(C, 1, 1, 1)
        k_hl = self.k_hl.repeat(C, 1, 1, 1)
        k_hh = self.k_hh.repeat(C, 1, 1, 1)

        ll = F.conv2d(x, k_ll, stride=2, padding=0, groups=C)
        lh = F.conv2d(x, k_lh, stride=2, padding=0, groups=C)
        hl = F.conv2d(x, k_hl, stride=2, padding=0, groups=C)
        hh = F.conv2d(x, k_hh, stride=2, padding=0, groups=C)

        return ll, lh, hl, hh

#-------------------------------------------------------

class UpsampleBlock(nn.Module):
    """PixelShuffle upsampling. Supports x2, x3, x4, and x8."""
    def __init__(self, in_ch, scale=4):
        super().__init__()
        assert scale in (2, 3, 4, 8)
        layers = []
        if scale in (2, 4, 8):
            n_up = {2: 1, 4: 2, 8: 3}[scale]
            for _ in range(n_up):
                layers += [nn.Conv2d(in_ch, in_ch * 4, 3, 1, 1), nn.PixelShuffle(2), nn.ReLU(inplace=True)]
        elif scale == 3:
            layers += [nn.Conv2d(in_ch, in_ch * 9, 3, 1, 1), nn.PixelShuffle(3), nn.ReLU(inplace=True)]
        self.body = nn.Sequential(*layers)

    def forward(self, x):
        return self.body(x)

#--------------------------------------------------------------
class MambaBlock2D(nn.Module):
    """
    2D -> sequence -> Mamba -> 2D
    input/output: (N, C, H, W)
    """
    def __init__(self, dim: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.mamba = Mamba(
            d_model=dim,
            d_state=d_state,
            d_conv=d_conv,
            expand=expand
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (N, C, H, W)
        n, c, h, w = x.shape
        residual = x

        # -> (N, H*W, C)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        x = self.mamba(x)

        # -> (N, C, H, W)
        x = x.transpose(1, 2).reshape(n, c, h, w)

        return x + residual


class MambaBranchB(nn.Module):
    """
    Path B:
    Receives all wavelet sub-bands jointly: LL, LH, HL, HH.
    Mamba models global dependencies over the joint wavelet representation.
    Then four separate heads produce sub-band-specific features:
    LL_B, LH_B, HL_B, HH_B.
    """
    def __init__(self, in_ch: int = 4, embed_dim: int = 96, num_blocks: int = 6):
        super().__init__()

        self.conv_first = nn.Conv2d(in_ch, embed_dim, kernel_size=3, stride=1, padding=1)

        blocks = []
        for _ in range(num_blocks):
            blocks.append(MambaBlock2D(dim=embed_dim, d_state=16, d_conv=4, expand=2))
        self.blocks = nn.Sequential(*blocks)

        self.conv_after = nn.Conv2d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1)

        # Separate output heads for each wavelet sub-band
        self.ll_head = nn.Conv2d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1)
        self.lh_head = nn.Conv2d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1)
        self.hl_head = nn.Conv2d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1)
        self.hh_head = nn.Conv2d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1)

    def forward(self, x: torch.Tensor):
        # x: (N,4,H/2,W/2)
        shallow = self.conv_first(x)
        feat = self.blocks(shallow)
        feat = self.conv_after(feat) + shallow

        feat_ll = self.ll_head(feat)
        feat_lh = self.lh_head(feat)
        feat_hl = self.hl_head(feat)
        feat_hh = self.hh_head(feat)

        return feat_ll, feat_lh, feat_hl, feat_hh
#-----------------------------------------
class ConvResBlock(nn.Module):
    """
    Simple residual convolutional block for sub-band feature extraction
    """
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, 1, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.block(x)


#--------------------------------------------------------------
class SingleBandCNNBranch(nn.Module):
    """
    CNN feature extractor for one wavelet sub-band.
    """
    def __init__(self, in_ch: int = 1, embed_dim: int = 96, num_blocks: int = 3):
        super().__init__()

        self.head = nn.Conv2d(in_ch, embed_dim, kernel_size=3, stride=1, padding=1)

        blocks = [ConvResBlock(embed_dim) for _ in range(num_blocks)]
        self.blocks = nn.Sequential(*blocks)

        self.tail = nn.Conv2d(embed_dim, embed_dim, kernel_size=3, stride=1, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shallow = self.head(x)
        feat = self.blocks(shallow)
        feat = self.tail(feat) + shallow
        return feat


class FullWaveletBranchA(nn.Module):
    """
    Path A:
    Processes each wavelet sub-band separately using CNN branches.
    Outputs four sub-band-specific feature maps:
    LL_A, LH_A, HL_A, HH_A.
    """
    def __init__(self, embed_dim: int = 96, num_blocks_ll: int = 4, num_blocks_hf: int = 3):
        super().__init__()

        self.ll_branch = SingleBandCNNBranch(
            in_ch=1,
            embed_dim=embed_dim,
            num_blocks=num_blocks_ll
        )

        self.lh_branch = SingleBandCNNBranch(
            in_ch=1,
            embed_dim=embed_dim,
            num_blocks=num_blocks_hf
        )

        self.hl_branch = SingleBandCNNBranch(
            in_ch=1,
            embed_dim=embed_dim,
            num_blocks=num_blocks_hf
        )

        self.hh_branch = SingleBandCNNBranch(
            in_ch=1,
            embed_dim=embed_dim,
            num_blocks=num_blocks_hf
        )

    def forward(self, ll: torch.Tensor, lh: torch.Tensor, hl: torch.Tensor, hh: torch.Tensor):
        feat_ll = self.ll_branch(ll)
        feat_lh = self.lh_branch(lh)
        feat_hl = self.hl_branch(hl)
        feat_hh = self.hh_branch(hh)

        return feat_ll, feat_lh, feat_hl, feat_hh
#---------------------------------------------------------------------------------------------
class HFAlignmentModule(nn.Module):
    """
    Gated and stable HF alignment module.
    Aligns high-frequency features with low-frequency structural features
    before fusion, while controlling alignment strength.
    """
    def __init__(self, channels: int, init_scale: float = 0.2):
        super().__init__()

        self.feat = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )

        self.delta_head = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        self.gate_head  = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)

        # scalar learnable strength, starts small for stability
        self.align_scale = nn.Parameter(torch.tensor(init_scale, dtype=torch.float32))

    def forward(self, feat_hf: torch.Tensor, feat_lf: torch.Tensor) -> torch.Tensor:
        x = torch.cat([feat_hf, feat_lf], dim=1)
        h = self.feat(x)

        delta = self.delta_head(h)
        gate = torch.sigmoid(self.gate_head(h))

        scale = torch.clamp(self.align_scale, min=0.0, max=1.0)

        aligned = feat_hf + scale * gate * delta
        return aligned
#--------------------------------------------------------------
class RobustMaxFusion(nn.Module):
    """
    Robust max-like fusion module for wavelet sub-band features.
    It performs magnitude-based hard selection, confidence-guided reliability weighting,
    and residual refinement. The same module is used for LL, LH, HL, and HH fusion.
    """
    def __init__(self, channels: int):
        super().__init__()
        self.confidence = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1),
        )

    def forward(self, feat_a: torch.Tensor, feat_b: torch.Tensor) -> torch.Tensor:
        # magnitude-based hard selection
        mask = (feat_a.abs() >= feat_b.abs()).float()
        hard_max = mask * feat_a + (1.0 - mask) * feat_b

        # confidence gate suppresses unreliable responses
        conf = self.confidence(torch.cat([feat_a, feat_b], dim=1))

        fused = conf * hard_max
        fused = fused + self.refine(fused)
        return fused
#---------------------------------------------------------------

#-----------------------------------------------------------------------------------

# generator--------------

class CGMFCTSRGenerator(nn.Module):
    """
    CGMFCTSRGenerator:
    - Decomposes LR input using fixed Haar DWT into LL, LH, HL, HH.
    - Path A extracts band-wise CNN features from each sub-band.
    - Path B processes all four wavelet bands jointly using 2D Mamba blocks.
    - LL features are fused using RobustMaxFusion.
    - The fused LL representation guides the alignment of Mamba high-frequency features.
    - LH, HL, and HH are aligned with the fused LL features and then fused using RobustMaxFusion.
    - Four fused bands are merged, upsampled by PixelShuffle, and added to bicubic LR residual.
    """
    def __init__(
        self,
        scale: int = 4,
        in_ch: int = 1,
        out_ch: int = 1,
        embed_dim: int = 96,
    ):
        super().__init__()
        self.scale = scale

        embed_dim_a = embed_dim
        embed_dim_b = embed_dim
        
        # --- Haar DWT ---
        self.dwt = HaarDWT2D()
        
        
        
        # === Two separate backbones ===
        self.branch_a = FullWaveletBranchA(
            embed_dim=embed_dim_a,
            num_blocks_ll=4,
            num_blocks_hf=3
        )
        self.mamba_b = MambaBranchB(
            in_ch=4,
            embed_dim=embed_dim_b,
            num_blocks=6
        )

        nf_a = embed_dim_a   
        self.pre_fusion_up = UpsampleBlock(nf_a, scale=2)

        self.ll_fusion = RobustMaxFusion(channels=nf_a)
        
        # Separate alignment modules for high-frequency bands
        self.lh_align = HFAlignmentModule(channels=nf_a, init_scale=0.25)
        self.hl_align = HFAlignmentModule(channels=nf_a, init_scale=0.25)
        self.hh_align = HFAlignmentModule(channels=nf_a, init_scale=0.1)
        
        # Separate robust max fusion for each high-frequency band
        self.lh_fusion = RobustMaxFusion(channels=nf_a)
        self.hl_fusion = RobustMaxFusion(channels=nf_a)
        self.hh_fusion = RobustMaxFusion(channels=nf_a)
        
        # Merge four fused sub-band features
        self.final_merge = nn.Sequential(
            nn.Conv2d(nf_a * 4, nf_a, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(nf_a, nf_a, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )

        
        # === Upsampling head ===
        self.ups  = UpsampleBlock(nf_a, scale=scale)
        self.tail = nn.Conv2d(nf_a, out_ch, 3, 1, 1)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: LR input (N,1,H,W) in [0,1]
        """
    
        # --------------------
        # Wavelet decomposition
        # --------------------
        ll, lh, hl, hh = self.dwt(x)   # each: (N,1,H/2,W/2)
    
        # --------------------
        # --------------------
        # Use raw wavelet bands (no normalization at all)
        # --------------------
        ll_n = ll
        lh_z = lh
        hl_z = hl
        hh_z = hh

    
        # --------------------
        # Path A: band-wise CNN features for LL, LH, HL, HH
        # --------------------
        feat_LL_A, feat_LH_A, feat_HL_A, feat_HH_A = self.branch_a(
            ll_n, lh_z, hl_z, hh_z
        )
        
        # --------------------
        # Path B: joint wavelet Mamba, then band-wise heads
        # --------------------
        x_b = torch.cat([ll_n, lh_z, hl_z, hh_z], dim=1)
        
        feat_LL_B, feat_LH_B, feat_HL_B, feat_HH_B = self.mamba_b(x_b)
        # Upsample both branches before fusion: 44x44 -> 88x88
        feat_LL_A = self.pre_fusion_up(feat_LL_A)
        feat_LH_A = self.pre_fusion_up(feat_LH_A)
        feat_HL_A = self.pre_fusion_up(feat_HL_A)
        feat_HH_A = self.pre_fusion_up(feat_HH_A)
        
        feat_LL_B = self.pre_fusion_up(feat_LL_B)
        feat_LH_B = self.pre_fusion_up(feat_LH_B)
        feat_HL_B = self.pre_fusion_up(feat_HL_B)
        feat_HH_B = self.pre_fusion_up(feat_HH_B)
        
        # --------------------
        # LL fusion using RobustMaxFusion
        # --------------------
        fused_LL = self.ll_fusion(feat_LL_A, feat_LL_B)
        
        # --------------------
        # Align Mamba high-frequency features using the fused LL structural representation
        # --------------------
        feat_LH_B_aligned = self.lh_align(feat_LH_B, fused_LL)
        feat_HL_B_aligned = self.hl_align(feat_HL_B, fused_LL)
        feat_HH_B_aligned = self.hh_align(feat_HH_B, fused_LL)
        
        # --------------------
        # Band-wise RobustMaxFusion for aligned high-frequency features
        # --------------------
        fused_LH = self.lh_fusion(feat_LH_A, feat_LH_B_aligned)
        fused_HL = self.hl_fusion(feat_HL_A, feat_HL_B_aligned)
        fused_HH = self.hh_fusion(feat_HH_A, feat_HH_B_aligned)
        
        # --------------------
        # Final band-wise merge
        # --------------------
        fused = torch.cat([fused_LL, fused_LH, fused_HL, fused_HH], dim=1)
        fused = self.final_merge(fused)
    
           
        # --------------------
        # Upsample + SR residual
        # --------------------
        out = self.tail(self.ups(fused))                                # (N,1,H*scale,W*scale)
    
        lr_up = F.interpolate(x, scale_factor=self.scale, mode="bicubic", align_corners=False)
        out = out + lr_up
    
        return out
#--------------------------------------------------------------
class PatchDiscriminator(nn.Module):
    """Patch discriminator used with LSGAN loss."""
    def __init__(self, in_ch=3, base_ch=64):
        super().__init__()

        # --- BEGIN: Discriminator Blocks ---
        def block(ic, oc, k=4, s=2, p=1):
            layers = [nn.Conv2d(ic, oc, k, s, p)]
           
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        # --- END ---

        layers = []
        ch = base_ch
        layers += block(in_ch, ch)
        layers += block(ch, ch*2)
        layers += block(ch*2, ch*4)
        layers += block(ch*4, ch*8)
        layers += [nn.Conv2d(ch*8, 1, kernel_size=4, stride=1, padding=0)]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)  # (B,1,h,w)
#--------------------------------------------------------------------------

#-------------------------------------------------------------------------------
# -----------------------------
# Losses
# -----------------------------

#------------------------------------------------------

def lsgan_g_loss(d_fake: torch.Tensor) -> torch.Tensor:
    """LSGAN generator loss: 0.5*(D(fake)-1)^2"""
    return 0.5 * F.mse_loss(d_fake, torch.ones_like(d_fake))
# ----------------------------------------------------------------
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, x, y):
        diff = x - y
        return torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))
#---------------------------------------------------------------------
class SSIMLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, y):
        if ssim is None:
            raise ImportError("Install pytorch-msssim")

        return 1.0 - ssim(
            x,
            y,
            data_range=1.0,
            size_average=True
        )
#----------------------------------------------------------------------

#----------------------------------------------

#--------------------------------------------------------


def _reflect_pad_if_needed(img, size):
    h, w = img.shape[:2]
    if h >= size and w >= size:
        return img
    pad_h = max(0, size - h)
    pad_w = max(0, size - w)
    return np.pad(
        img,
        ((pad_h//2, pad_h - pad_h//2),
         (pad_w//2, pad_w - pad_w//2)),
        mode="reflect"
    )
    
def _make_divisible_by_scale(img, scale):
        h, w = img.shape[:2]
        h2 = (h // scale) * scale
        w2 = (w // scale) * scale
        return img[:h2, :w2]
    
def _augment(lq, gt):
    if random.random() < 0.5:
        lq = np.fliplr(lq); gt = np.fliplr(gt)
    if random.random() < 0.5:
        lq = np.flipud(lq); gt = np.flipud(gt)
    if random.random() < 0.5:
        k = random.choice([1, 2, 3])
        lq = np.rot90(lq, k); gt = np.rot90(gt, k)
    return lq.copy(), gt.copy()

# -----------------------------
# Dataset (patch-based SR pairs)
# -----------------------------

class ImagePairDataset(Dataset):
    """
    On-the-fly SR pair generation from GT images:
    - Read GT (grayscale)
    - reflect-pad if needed (>= HR)
    - make divisible by scale
    - bicubic downsample FULL image -> lr_full
    - crop LR patch (random for train, center for val)
    - crop matched HR patch from GT using (top_lr*scale, left_lr*scale)
    """
    def __init__(self, root: str, scale: int = 4, lr_size: int = 88,
                 exts=(".png", ".jpg", ".jpeg", ".tif", ".tiff"), training=True,
                 use_hflip=True, use_rot=True):
        super().__init__()
        self.files = []
        for dp, _, fns in os.walk(root):
            for f in fns:
                if os.path.splitext(f)[1].lower() in exts:
                    self.files.append(os.path.join(dp, f))
        self.files.sort()

        self.scale = scale
        self.lr_size = lr_size
        self.hr_size = lr_size * scale
        self.training = training
        self.use_hflip = use_hflip
        self.use_rot = use_rot

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx: int):
        gt_path = self.files[idx]
        gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
        if gt is None:
            raise FileNotFoundError(gt_path)

        # 1) pad if smaller than HR patch
        gt_pad = _reflect_pad_if_needed(gt, self.hr_size)

        # 2) make divisible by scale
        gt_use = _make_divisible_by_scale(gt_pad, self.scale)
        H, W = gt_use.shape[:2]

        # 3) full-image bicubic downsample -> LR_full
        lr_full = cv2.resize(
            gt_use,
            (W // self.scale, H // self.scale),  # (width, height)
            interpolation=cv2.INTER_CUBIC
        )

        # 4) crop LR (random for train, center for val)
        h_lr, w_lr = lr_full.shape[:2]
        if self.training:
            top_lr = random.randint(0, h_lr - self.lr_size)
            left_lr = random.randint(0, w_lr - self.lr_size)
        else:
            top_lr = (h_lr - self.lr_size) // 2
            left_lr = (w_lr - self.lr_size) // 2

        lr = lr_full[top_lr:top_lr + self.lr_size, left_lr:left_lr + self.lr_size]

        # 5) crop matched HR from GT
        top_hr = top_lr * self.scale
        left_hr = left_lr * self.scale
        hr = gt_use[top_hr:top_hr + self.hr_size, left_hr:left_hr + self.hr_size]

        # sanity
        assert hr.shape == (self.hr_size, self.hr_size), \
            f"HR size mismatch: got {hr.shape}, expected ({self.hr_size},{self.hr_size}) | {gt_path}"
        assert lr.shape == (self.lr_size, self.lr_size), \
            f"LR size mismatch: got {lr.shape}, expected ({self.lr_size},{self.lr_size}) | {gt_path}"

        # 6) augmentation (train only)
        if self.training and (self.use_hflip or self.use_rot):
            lr, hr = _augment(lr, hr)

        # to tensor in [0,1], shape (1,H,W)
        hr_t = torch.from_numpy(hr.astype(np.float32) / 255.0).unsqueeze(0)
        lr_t = torch.from_numpy(lr.astype(np.float32) / 255.0).unsqueeze(0)

        return lr_t, hr_t


# -----------------------------
# Training / Validation loops
# -----------------------------

@dataclass
class TrainConfig:
    train_dir: str
    val_dir: str
    save_dir: str = './runs/Module'
    scale: int = 4
    lr_size: int = 88
    batch_size: int = 8
    num_workers: int = 0
    epochs: int = 100
    lr_g: float = 2e-4
    lr_d: float = 1e-4
    beta1: float = 0.5
    beta2: float = 0.999
    adv_weight: float = 0.0008
    content_weight: float = 1.0
    ssim_weight: float = 0.15
    disc_channels: int = 64
    w_charb: float = 1.15
    ema_decay: float = 0.999
    gan_warmup_epochs: int = 1
    

    device: torch.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



def validate(gen, loader, device='cuda'):
    gen.eval()
    all_psnr, all_ssim = [], []
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(device), hr.to(device)
            sr = gen(lr).clamp(0, 1)
            p, s = batch_psnr_ssim_y(sr, hr)
            all_psnr.append(p)
            all_ssim.append(s)
    return float(np.mean(all_psnr)), float(np.mean(all_ssim))


def train(cfg: TrainConfig):
    seed = 123
    seed_everything(seed, deterministic=False)

    g = torch.Generator()
    g.manual_seed(seed)
    
    # ----------------------------------------
    if torch.cuda.is_available():
        device_name = torch.cuda.get_device_name(0)
        cuda_ver = torch.version.cuda
        print("GPU Detected:", device_name, "- using CUDA", cuda_ver)
    else:
        print("No GPU detected. Training will run on CPU.")

    os.makedirs(cfg.save_dir, exist_ok=True)

    # ===================== Datasets & Loaders =====================
    train_set = ImagePairDataset(cfg.train_dir, scale=cfg.scale, lr_size=cfg.lr_size, training=True,
                             use_hflip=True, use_rot=True)
    val_set   = ImagePairDataset(cfg.val_dir,   scale=cfg.scale, lr_size=cfg.lr_size, training=False,
                             use_hflip=False, use_rot=False)


    train_loader = DataLoader(
        train_set,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
        generator=g
    )

    lr, hr = next(iter(train_loader))
    print("LR shape:", lr.shape)
    print("HR shape:", hr.shape)
    assert lr.shape[-2:] == (cfg.lr_size, cfg.lr_size)
    assert hr.shape[-2:] == (cfg.lr_size*cfg.scale, cfg.lr_size*cfg.scale)

    # ===================== Sanity check: Haar DWT + CNN/Mamba dual-branch forward =====================
    with torch.no_grad():
        lr_dbg = lr.to(cfg.device)

        # check 
        gen_tmp = CGMFCTSRGenerator(
            scale=cfg.scale,
            in_ch=1, out_ch=1,            
            embed_dim=96            
        ).to(cfg.device)

        print("mamba_b in_ch:", gen_tmp.mamba_b.conv_first.in_channels)

        ll_dbg, lh_dbg, hl_dbg, hh_dbg = gen_tmp.dwt(lr_dbg)
        print("LL:", tuple(ll_dbg.shape), "LH:", tuple(lh_dbg.shape), "HL:", tuple(hl_dbg.shape), "HH:", tuple(hh_dbg.shape))
        
        ll_n_dbg = ll_dbg
        lh_z_dbg = lh_dbg
        hl_z_dbg = hl_dbg
        hh_z_dbg = hh_dbg

        feat_LL_A_dbg, feat_LH_A_dbg, feat_HL_A_dbg, feat_HH_A_dbg = gen_tmp.branch_a(
            ll_n_dbg, lh_z_dbg, hl_z_dbg, hh_z_dbg
        )
        
        print("Path A LL shape:", tuple(feat_LL_A_dbg.shape))
        print("Path A LH shape:", tuple(feat_LH_A_dbg.shape))
        print("Path A HL shape:", tuple(feat_HL_A_dbg.shape))
        print("Path A HH shape:", tuple(feat_HH_A_dbg.shape))
        
        x_b_dbg = torch.cat([ll_n_dbg, lh_z_dbg, hl_z_dbg, hh_z_dbg], dim=1)
        print("Path B input shape:", tuple(x_b_dbg.shape), "| range:", float(x_b_dbg.min()), float(x_b_dbg.max()))
        
        feat_LL_B_dbg, feat_LH_B_dbg, feat_HL_B_dbg, feat_HH_B_dbg = gen_tmp.mamba_b(x_b_dbg)
        feat_LL_A_dbg = gen_tmp.pre_fusion_up(feat_LL_A_dbg)
        feat_LH_A_dbg = gen_tmp.pre_fusion_up(feat_LH_A_dbg)
        feat_HL_A_dbg = gen_tmp.pre_fusion_up(feat_HL_A_dbg)
        feat_HH_A_dbg = gen_tmp.pre_fusion_up(feat_HH_A_dbg)
        
        feat_LL_B_dbg = gen_tmp.pre_fusion_up(feat_LL_B_dbg)
        feat_LH_B_dbg = gen_tmp.pre_fusion_up(feat_LH_B_dbg)
        feat_HL_B_dbg = gen_tmp.pre_fusion_up(feat_HL_B_dbg)
        feat_HH_B_dbg = gen_tmp.pre_fusion_up(feat_HH_B_dbg)
        
        print("Path A LL after pre-fusion up:", tuple(feat_LL_A_dbg.shape))
        print("Path B LL after pre-fusion up:", tuple(feat_LL_B_dbg.shape))
        
        print("Path B LL shape:", tuple(feat_LL_B_dbg.shape))
        print("Path B LH shape:", tuple(feat_LH_B_dbg.shape))
        print("Path B HL shape:", tuple(feat_HL_B_dbg.shape))
        print("Path B HH shape:", tuple(feat_HH_B_dbg.shape))
        
        fused_LL_dbg = gen_tmp.ll_fusion(feat_LL_A_dbg, feat_LL_B_dbg)
        
        feat_LH_B_aligned_dbg = gen_tmp.lh_align(feat_LH_B_dbg, fused_LL_dbg)
        feat_HL_B_aligned_dbg = gen_tmp.hl_align(feat_HL_B_dbg, fused_LL_dbg)
        feat_HH_B_aligned_dbg = gen_tmp.hh_align(feat_HH_B_dbg, fused_LL_dbg)
        
        fused_LH_dbg = gen_tmp.lh_fusion(feat_LH_A_dbg, feat_LH_B_aligned_dbg)
        fused_HL_dbg = gen_tmp.hl_fusion(feat_HL_A_dbg, feat_HL_B_aligned_dbg)
        fused_HH_dbg = gen_tmp.hh_fusion(feat_HH_A_dbg, feat_HH_B_aligned_dbg)
        
        print("Fused LL shape:", tuple(fused_LL_dbg.shape))
        print("Fused LH shape:", tuple(fused_LH_dbg.shape))
        print("Fused HL shape:", tuple(fused_HL_dbg.shape))
        print("Fused HH shape:", tuple(fused_HH_dbg.shape))
        
        fused_dbg = torch.cat([fused_LL_dbg, fused_LH_dbg, fused_HL_dbg, fused_HH_dbg], dim=1)
        print("Concat fused bands shape:", tuple(fused_dbg.shape))
        
        fused_dbg = gen_tmp.final_merge(fused_dbg)
        print("Final merged feature shape:", tuple(fused_dbg.shape))
        


        sr_dbg = gen_tmp(lr_dbg)
        print("SR shape:", tuple(sr_dbg.shape), " | range:", float(sr_dbg.min()), float(sr_dbg.max()))

        # (N,1, lr_size*scale, lr_size*scale)
        assert sr_dbg.shape[-2:] == (cfg.lr_size*cfg.scale, cfg.lr_size*cfg.scale), \
            f"SR size mismatch: got {sr_dbg.shape[-2:]}"


    val_loader = DataLoader(
        val_set,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True
    )

    # ===================== Models =====================
    gen = CGMFCTSRGenerator(
        scale=cfg.scale,
        in_ch=1, out_ch=1,
        embed_dim=96        
    ).to(cfg.device)

       
    ema = EMA(gen, decay=cfg.ema_decay)
    
    disc_patch = PatchDiscriminator(in_ch=1, base_ch=cfg.disc_channels).to(cfg.device)


    # ===================== Losses & Optimizers =====================
    
    charb = CharbonnierLoss().to(cfg.device)
    ssim_loss_func = SSIMLoss().to(cfg.device)


    opt_g = torch.optim.Adam(gen.parameters(), lr=cfg.lr_g, betas=(cfg.beta1, cfg.beta2))
    opt_d_patch = torch.optim.Adam(disc_patch.parameters(), lr=cfg.lr_d, betas=(cfg.beta1, cfg.beta2))


    # =====================Resume =====================
    checkpoint_path = os.path.join(cfg.save_dir, 'last.pt')
    start_epoch = 1 
    best_psnr = -1.0 
    
    if os.path.exists(checkpoint_path):
        print(f" Loading checkpoint from {checkpoint_path}...")
        
        checkpoint = torch.load(checkpoint_path, map_location=cfg.device) 
        
        gen.load_state_dict(checkpoint['gen'])

        if 'disc_patch' in checkpoint:
            disc_patch.load_state_dict(checkpoint['disc_patch'])
                
        if 'opt_g' in checkpoint:
            opt_g.load_state_dict(checkpoint['opt_g'])
        if 'opt_d_patch' in checkpoint:
            opt_d_patch.load_state_dict(checkpoint['opt_d_patch'])
        

        
        start_epoch = checkpoint.get('epoch', 0) + 1 
        best_psnr = checkpoint.get('best_psnr', -1.0)
        
        print(f" Training resumed from Epoch {start_epoch}. Current best PSNR: {best_psnr:.4f}")
# =================================================================

    log_path = os.path.join(cfg.save_dir, 'log.txt')
        
    # ===================== CSV: debug per-iter =====================
    debug_path = os.path.join(cfg.save_dir, 'train_debug_iter.csv')
    if not os.path.exists(debug_path) and start_epoch == 1:
        with open(debug_path, 'w', encoding='utf-8') as f:
            f.write(
                "epoch,iter,loss_g,loss_d,content,adv,"
                "sr_min,sr_max,sr_mean,lr_min,lr_max,hr_min,hr_max\n"
            )

    # ===================== CSV: train =====================
    train_loss_path = os.path.join(cfg.save_dir, 'train_loss_epoch.csv')
    if not os.path.exists(train_loss_path) and start_epoch == 1:
        with open(train_loss_path, 'w', encoding='utf-8') as f:
           f.write("epoch,loss_g,loss_d,content,adv,psnr,ssim\n")


    # ===================== CSV: val =====================
    val_metrics_path = os.path.join(cfg.save_dir, 'val_metrics_epoch.csv')
    if not os.path.exists(val_metrics_path) and start_epoch == 1:
        with open(val_metrics_path, 'w', encoding='utf-8') as f:
            f.write("epoch,psnr,ssim\n")


    # ===================== Epoch Loop =====================
    
    for epoch in range(start_epoch, cfg.epochs + 1):
    
        gen.train()
        disc_patch.train()

        epoch_loss_g = 0.0
        epoch_loss_d = 0.0
        epoch_content = 0.0
        epoch_adv = 0.0
        epoch_train_psnr = 0.0
        epoch_train_ssim = 0.0
        num_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs}")
        for lr, hr in pbar:
            lr, hr = lr.to(cfg.device), hr.to(cfg.device)

            use_gan = (epoch > cfg.gan_warmup_epochs)
            
            # ===== 1) Train Discriminator (LSGAN) =====
            if use_gan:
                with torch.no_grad():
                    sr_detached = gen(lr).clamp(0, 1)
            
                hr_d = hr.clamp(0, 1)
            
                # ---- Patch Discriminator ----
                d_real_p = disc_patch(hr_d)
                d_fake_p = disc_patch(sr_detached)
            
                loss_d_patch = 0.5 * (
                    F.mse_loss(d_real_p, torch.ones_like(d_real_p)) +
                    F.mse_loss(d_fake_p, torch.zeros_like(d_fake_p))
                )
            
                opt_d_patch.zero_grad(set_to_none=True)
                loss_d_patch.backward()
                opt_d_patch.step()
            
                           
                loss_d = loss_d_patch 
            else:
                loss_d = torch.tensor(0.0, device=cfg.device)

            
            # ===== 2) Train Generator =====
            sr = gen(lr)
            
            sr_c = sr.clamp(0, 1)
            hr_c = hr.clamp(0, 1)
            
            loss_charb = charb(sr_c, hr_c)
            loss_ssim = ssim_loss_func(sr_c, hr_c)            
            
                        
            loss_content = (
                (cfg.w_charb * loss_charb) +
                (cfg.ssim_weight * loss_ssim)
            )

            
            if use_gan:
                d_fake_p = disc_patch(sr_c)
            
                loss_adv = lsgan_g_loss(d_fake_p)
            else:
                loss_adv = torch.tensor(0.0, device=cfg.device)

            
            loss_g = (
                cfg.content_weight    * loss_content +
                cfg.adv_weight        * loss_adv
            )

            # ===== Debug stats (SR range) =====
            with torch.no_grad():
                sr_min  = float(sr.min().item())
                sr_max  = float(sr.max().item())
                sr_mean = float(sr.mean().item())

                lr_min = float(lr.min().item())
                lr_max = float(lr.max().item())
                hr_min = float(hr.min().item())
                hr_max = float(hr.max().item())

            iter_in_epoch = num_batches + 1

            with open(debug_path, 'a', encoding='utf-8') as f:
                f.write(
                    f"{epoch},{iter_in_epoch},"
                    f"{loss_g.item():.6f},{loss_d.item():.6f},"
                    f"{loss_content.item():.6f},"
                    f"{loss_adv.item():.6f},"
                    f"{sr_min:.6f},{sr_max:.6f},{sr_mean:.6f},"
                    f"{lr_min:.6f},{lr_max:.6f},{hr_min:.6f},{hr_max:.6f}\n"
                )
            # ===================== Stability guard =====================
            if not torch.isfinite(loss_g):
                print(f" Non-finite loss_g at epoch {epoch}, iter {iter_in_epoch}. Skipping step.")
                opt_g.zero_grad(set_to_none=True)
                continue

            if (sr_min < -1.0) or (sr_max > 2.0):
                print(f" SR out-of-range at epoch {epoch}, iter {iter_in_epoch}: "
                      f"min={sr_min:.3f}, max={sr_max:.3f}")

                       
            opt_g.zero_grad(set_to_none=True)
            loss_g.backward()
            

            torch.nn.utils.clip_grad_norm_(gen.parameters(), max_norm=1.0)
            opt_g.step()
            ema.update(gen)

            # ===== 3) Accumulate losses + train PSNR/SSIM =====
            epoch_loss_d      += loss_d.item()
            epoch_loss_g      += loss_g.item()
            epoch_content     += loss_content.item()
            epoch_adv         += loss_adv.item()


            with torch.no_grad():
                p, s = batch_psnr_ssim_y(sr.clamp(0, 1), hr)
                epoch_train_psnr += p
                epoch_train_ssim += s

            num_batches += 1

            pbar.set_postfix({
                'loss_d': f"{loss_d.item():.3f}",
                'loss_g': f"{loss_g.item():.3f}",
                'content': f"{loss_content.item():.3f}",
                'ssim': f"{loss_ssim.item():.3f}",
                'adv': f"{loss_adv.item():.3f}",
            })

        # ===== 4) Averages per epoch (train) =====
        mean_loss_g      = epoch_loss_g      / max(1, num_batches)
        mean_loss_d      = epoch_loss_d      / max(1, num_batches)
        mean_content     = epoch_content     / max(1, num_batches)
        mean_adv         = epoch_adv         / max(1, num_batches)
        mean_train_psnr  = epoch_train_psnr  / max(1, num_batches)
        mean_train_ssim  = epoch_train_ssim  / max(1, num_batches)


        with open(train_loss_path, 'a', encoding='utf-8') as f:
            f.write(
                f"{epoch},{mean_loss_g:.6f},{mean_loss_d:.6f},"
                f"{mean_content:.6f},{mean_adv:.6f},"
                f"{mean_train_psnr:.6f},{mean_train_ssim:.6f}\n"

            )

        print(
            f"[TRAIN] Epoch {epoch}: "
            f"PSNR={mean_train_psnr:.3f}, SSIM={mean_train_ssim:.3f} "
            f"loss_g={mean_loss_g:.4f}, loss_d={mean_loss_d:.4f}, "
            f"content={mean_content:.4f}, "
            f"adv={mean_adv:.4f}"
        )

        # ===== 5) Validation (using EMA weights) =====
        ema.store(gen)
        ema.copy_to(gen)
        val_psnr, val_ssim = validate(gen, val_loader, cfg.device)
        ema.restore(gen)

        with open(val_metrics_path, 'a', encoding='utf-8') as f_val:
            f_val.write(f"{epoch},{val_psnr:.6f},{val_ssim:.6f}\n")

        
        with open(log_path, 'a', encoding='utf-8') as f:
            f.write(f"{epoch}\t{val_psnr:.3f}\t{val_ssim:.3f}\n")

        # ===== 6) (last + best) =====
        
        ema.store(gen)
        ema.copy_to(gen)
        torch.save(
            {
                'gen': gen.state_dict(),
                'disc_patch': disc_patch.state_dict(),
                'cfg': cfg.__dict__,
                'opt_g': opt_g.state_dict(),
                'opt_d_patch': opt_d_patch.state_dict(),               
                'epoch': epoch,
                'best_psnr': best_psnr,
            },
            os.path.join(cfg.save_dir, 'last.pt')
        )
        ema.restore(gen)


        if val_psnr > best_psnr:
            best_psnr = val_psnr
            ema.store(gen)
            ema.copy_to(gen)
            torch.save(
            {
                'gen': gen.state_dict(),
                'disc_patch': disc_patch.state_dict(),
                'cfg': cfg.__dict__,
                'opt_g': opt_g.state_dict(),
                'opt_d_patch': opt_d_patch.state_dict(),
                'epoch': epoch,
                'best_psnr': best_psnr,
            },
            os.path.join(cfg.save_dir, 'best.pt')
        )
            ema.restore(gen)

        print(
            f"[VAL] Epoch {epoch}: PSNR={val_psnr:.3f}, SSIM={val_ssim:.3f} "
            f"(best PSNR={best_psnr:.3f})"
        )
'''
Module_code = Module_code.replace(u'\xa0', u' ')
with open("Module.py", "w", encoding="utf-8") as f:
    f.write(Module_code) 
print(" Module.py is created")

In [ ]:
# ==========================
# Model Parameters Counter
# ==========================

import torch
import inspect
import Module

device = "cuda" if torch.cuda.is_available() else "cpu"


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("=" * 60)
    print(f"Model : {model.__class__.__name__}")
    print("=" * 60)
    print(f"Total Parameters      : {total:,}")
    print(f"Trainable Parameters  : {trainable:,}")
    print(f"Parameters (Million)  : {total / 1e6:.3f} M")
    print("=" * 60)

    return total, trainable


# ---------------------------------------------------
# Automatically find the generator class
# ---------------------------------------------------
generator_cls = None

for name, obj in inspect.getmembers(Module, inspect.isclass):
    if (
        issubclass(obj, torch.nn.Module)
        and "Generator" in name
        and obj.__module__ == "Module"
    ):
        generator_cls = obj
        break

if generator_cls is None:
    raise RuntimeError("No generator class found in Module.py")

print(f"Generator found: {generator_cls.__name__}")

model = generator_cls(
    scale=4,
    out_ch=1,
    embed_dim=96
).to(device)

count_parameters(model)

In [ ]:
# Training------
from Module import TrainConfig, train

cfg = TrainConfig(
    train_dir="path of images to train",
    val_dir="path of images to validate",
    save_dir="results path",
    scale=4,
    lr_size=88,
    batch_size=4,
    num_workers=0,
    epochs=30,
)
train(cfg)


In [ ]:
#--------Test the model-------#
import os, glob, warnings, sys
import cv2
import torch
import numpy as np
import pandas as pd
import lpips

# ====== BasicSR path (WSL path) ======
sys.path.insert(0, "/mnt/c/Users/lenovo/BasicSR")
from basicsr.metrics.psnr_ssim import calculate_psnr, calculate_ssim

warnings.filterwarnings("ignore", message="torch.meshgrid: in an upcoming release*")
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")

# ====== DISTS ======
dists_fn = None
try:
    from DISTS_pytorch import DISTS
    dists_fn = DISTS().eval()
except Exception:
    try:
        from dists_pytorch import DISTS
        dists_fn = DISTS().eval()
    except Exception as e:
        print("error", e)

# ====== LPIPS ======
lpips_fn = lpips.LPIPS(net="alex").eval()

def to_3ch_minus1_1(img01_hw):
    """img in [0,1] shape (H,W) -> torch (1,3,H,W) in [-1,1]"""
    t = torch.from_numpy(img01_hw).float()[None, None, ...]  # (1,1,H,W)
    t = t.repeat(1, 3, 1, 1)                                  # (1,3,H,W)
    t = t * 2.0 - 1.0
    return t

# ====== Model ======
from Module import CGMFCTSRGenerator

# ========= Paths (WSL) =========
lr_dir   = "LR images path for test"
gt_dir   = "HR images path for test"
out_dir  = "path for test results"
csv_path = os.path.join(out_dir, "metrics_per_image.csv")

ckpt_path = "path of weights file /best.pt"

os.makedirs(out_dir, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ====== Load checkpoint ======
ckpt = torch.load(ckpt_path, map_location=device)
cfg = ckpt.get("cfg", {})

scale = int(cfg.get("scale", 4))
lr_size = int(cfg.get("lr_size", 88))

net = CGMFCTSRGenerator(
    scale=scale,
    in_ch=1,
    out_ch=1,
    embed_dim=96
).to(device).eval()


net.load_state_dict(ckpt["gen"], strict=True)
print("Loaded weights strictly (architecture matches).")

# ====== Architecture check for band-wise fusion model ======

if hasattr(net, "branch_a"):
    print(" Path A is present.")
    print("   LL branch out channels:", net.branch_a.ll_branch.head.out_channels)
    print("   LH branch out channels:", net.branch_a.lh_branch.head.out_channels)
    print("   HL branch out channels:", net.branch_a.hl_branch.head.out_channels)
    print("   HH branch out channels:", net.branch_a.hh_branch.head.out_channels)
else:
    print("Path A is NOT present.")

if hasattr(net, "mamba_b"):
    print("Path B (Mamba) is present.")
    print("   Mamba input channels:", net.mamba_b.conv_first.in_channels)
    print("   Mamba first conv out channels:", net.mamba_b.conv_first.out_channels)
    print("   LL_B head out channels:", net.mamba_b.ll_head.out_channels)
    print("   LH_B head out channels:", net.mamba_b.lh_head.out_channels)
    print("   HL_B head out channels:", net.mamba_b.hl_head.out_channels)
    print("   HH_B head out channels:", net.mamba_b.hh_head.out_channels)
else:
    print(" Path B (Mamba) is NOT present.")

if hasattr(net, "ll_fusion"):
    print(" LL fusion module is present:", type(net.ll_fusion).__name__)
else:
    print(" LL fusion module is NOT present.")

if hasattr(net, "lh_fusion") and hasattr(net, "hl_fusion") and hasattr(net, "hh_fusion"):
    print(" Band-wise HF fusion modules are present.")
else:
    print(" Band-wise HF fusion modules are NOT complete.")

if hasattr(net, "lh_align") and hasattr(net, "hl_align") and hasattr(net, "hh_align"):
    print(" Band-wise HF alignment modules are present.")
else:
    print(" Band-wise HF alignment modules are NOT complete.")

if hasattr(net, "final_merge"):
    print(" Final merge module is present.")
    print("   Final merge input channels:", net.final_merge[0].in_channels)
    print("   Final merge output channels:", net.final_merge[0].out_channels)
else:
    print(" Final merge module is NOT present.")

print("Model loaded:", os.path.basename(ckpt_path))

lpips_fn = lpips_fn.to(device)
if dists_fn is not None:
    dists_fn = dists_fn.to(device)

def list_images(folder):
    exts = ("*.png","*.jpg","*.jpeg","*.tif","*.tiff")
    files = []
    for e in exts:
        files += glob.glob(os.path.join(folder, e))
    return sorted(files)

lr_files = list_images(lr_dir)
gt_files = list_images(gt_dir)

lr_map = {os.path.basename(p): p for p in lr_files}
gt_map = {os.path.basename(p): p for p in gt_files}
common = sorted(set(lr_map.keys()) & set(gt_map.keys()))
print("Common pairs:", len(common))

rows = []

with torch.no_grad():
    for name in common:
        lr_path = lr_map[name]
        gt_path = gt_map[name]

        lr = cv2.imread(lr_path, cv2.IMREAD_GRAYSCALE)
        gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)

        if lr is None or gt is None:
            print("Skip (read fail):", name)
            continue

        if lr.dtype != np.uint8:
            lr = lr.astype(np.uint8)
        if gt.dtype != np.uint8:
            gt = gt.astype(np.uint8)

        if lr.shape != (88, 88) or gt.shape != (352, 352):
            print(f"unexpected {name}: LR{lr.shape}, GT{gt.shape} — ")
            continue

        lr01 = lr.astype(np.float32) / 255.0
        gt01 = gt.astype(np.float32) / 255.0

        lr_t = torch.from_numpy(lr01)[None, None, ...].to(device)
        sr_t = net(lr_t).clamp(0, 1)
        sr01 = sr_t.squeeze().cpu().numpy()

        # save SR
        sr_u8 = (sr01 * 255.0).round().clip(0,255).astype(np.uint8)
        cv2.imwrite(os.path.join(out_dir, name), sr_u8)

        # metrics
        psnr = calculate_psnr(sr_u8, gt, crop_border=0, test_y_channel=False)
        ssim = calculate_ssim(sr_u8, gt, crop_border=0, test_y_channel=False)

        sr_lp = to_3ch_minus1_1(sr01).to(device)
        gt_lp = to_3ch_minus1_1(gt01).to(device)

        lp = float(lpips_fn(sr_lp, gt_lp).item())
        if dists_fn is not None:
            ds = float(dists_fn(sr_lp, gt_lp).item())
        else:
            ds = np.nan

        rows.append({
            "name": name,
            "psnr": psnr,
            "ssim": ssim,
            "lpips": lp,
            "dists": ds
        })

df = pd.DataFrame(rows)

summary = pd.DataFrame([
    {
        "name": "MEAN",
        "psnr": df["psnr"].mean(),
        "ssim": df["ssim"].mean(),
        "lpips": df["lpips"].mean(),
        "dists": df["dists"].mean(),
    },
    {
        "name": "STD",
        "psnr": df["psnr"].std(ddof=1),
        "ssim": df["ssim"].std(ddof=1),
        "lpips": df["lpips"].std(ddof=1),
        "dists": df["dists"].std(ddof=1),
    }
])

out_df = pd.concat([df, summary], ignore_index=True)
out_df.to_csv(csv_path, index=False)

print("Saved SR images to:", out_dir)
print("Saved CSV to:", csv_path)
display(out_df.tail(10))